# 00. PyTorch Fundamentals

This notebook covers the fundamental building blocks of PyTorch: tensors — how to create them, inspect their properties, reshape/manipulate them, and run basic operations on them.

**Resources**
1. Notebook: https://www.learnpytorch.io/00_pytorch_fundamentals/
2. Video: https://www.youtube.com/watch?v=V_xro1bcAuA&t=5277s


## Setup & Environment Check

Import the required libraries and check which device (CPU or GPU/ROCm) is available, so later tensor operations can be placed on the right device.


In [68]:
# Capture the runtime details so we know which image-processing paths are available.
import platform
import torch
import torch.nn.functional as F
import warnings
import torch
import numpy as np
import matplotlib.pyplot as plt

# Silence warnings so the notebook output stays focused on the examples.
warnings.filterwarnings("ignore")

print(f"--- System Information ---")
print(f"Platform: {platform.platform()}")
print(f"Python:   {platform.python_version()}")
print(f"PyTorch:  {torch.__version__}")

print(f"\n--- GPU/ROCm Accelerators ---")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    print(f"Device Count:   {torch.cuda.device_count()}")
    print(f"Primary Device: {torch.cuda.get_device_name(0)}")
    DEVICE = torch.device("cuda:0")
else:
    print("Optimization Note: No CUDA-capable GPU detected. Operations will run on CPU.")
    DEVICE = torch.device("cpu")


--- System Information ---
Platform: Windows-11-10.0.26200-SP0
Python:   3.12.13
PyTorch:  2.9.1+rocm7.2.1

--- GPU/ROCm Accelerators ---
CUDA Available: True
Device Count:   1
Primary Device: AMD Radeon RX 7900 XT


## Introduction to Tensors
### Creating Tensors

Tensors are the core data structure in PyTorch — multi-dimensional arrays similar to NumPy arrays, but with GPU acceleration and automatic differentiation support. Depending on their number of dimensions, they represent scalars, vectors, matrices, or higher-dimensional data.


In [69]:
# scalar tensor
# Scalar tensors are 0-dimensional arrays. They can be created from Python numbers or NumPy scalars.
scalar = torch.tensor(7)
print(f"Scalar: {scalar}\nShape: {scalar.shape}\nDimensions: {scalar.ndim}")
print(f"Scalar Value: {scalar.item()}")  # .item() extracts the value as a standard Python number

Scalar: 7
Shape: torch.Size([])
Dimensions: 0
Scalar Value: 7


In [70]:
# Vector tensor
# Vector tensors are 1-dimensional arrays. They can be created from Python lists or NumPy arrays.
vector = torch.tensor([7,7])
print(f"Vector: {vector}\nShape: {vector.shape}\nDimensions: {vector.ndim}")

Vector: tensor([7, 7])
Shape: torch.Size([2])
Dimensions: 1


In [71]:
# Matrix tensor
# Matrix tensors are 2-dimensional arrays. They can be created from nested Python lists or NumPy arrays.
matrix = torch.tensor([[7, 8]
                       ,[9, 10]])

print(f"Matrix: {matrix}\nShape: {matrix.shape}\nDimensions: {matrix.ndim}")

print(f"First row of the matrix: {matrix[0]}")
print(f"Second row of the matrix: {matrix[1]}")

Matrix: tensor([[ 7,  8],
        [ 9, 10]])
Shape: torch.Size([2, 2])
Dimensions: 2
First row of the matrix: tensor([7, 8])
Second row of the matrix: tensor([ 9, 10])


In [72]:
# Tensor
tensor = torch.tensor([[[1, 2], [3, 4]], 
                       [[5, 6], [7, 8]]])

print(f"Tensor: {tensor}, Shape: {tensor.shape}, Dimensions: {tensor.ndim}")

print(f"First slice of the tensor: {tensor[0]}")
print(f"Second slice of the tensor: {tensor[1]}")
print(f"First row of the first slice: {tensor[0][0]}")
print(f"Second row of the first slice: {tensor[0][1]}")

Tensor: tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]]), Shape: torch.Size([2, 2, 2]), Dimensions: 3
First slice of the tensor: tensor([[1, 2],
        [3, 4]])
Second slice of the tensor: tensor([[5, 6],
        [7, 8]])
First row of the first slice: tensor([1, 2])
Second row of the first slice: tensor([3, 4])


### Random Tensors
Random tensors are important because they are the basis of many random processes in machine learning, such as weight initialization in neural networks. In PyTorch, you can create random tensors using various functions.

In [73]:
# Random tensors are important in machine learning for initializing/adjusting weights and biases.
# They can be created using various distributions, such as uniform or normal distributions.

random_tensor = torch.rand(3, 4) # Creates a 3x4 tensor filled with random numbers from a uniform distribution over [0, 1)
print(f"Random Tensor: {random_tensor}\nShape: {random_tensor.shape}\nDimensions: {random_tensor.ndim}")

Random Tensor: tensor([[0.7088, 0.3651, 0.8843, 0.9785],
        [0.6797, 0.7685, 0.2305, 0.4446],
        [0.5726, 0.2790, 0.1099, 0.1131]])
Shape: torch.Size([3, 4])
Dimensions: 2


In [74]:
# Random tensor with similar shape to an image tensor (e.g., 3 channels, 224x224 pixels)
image_like_tensor = torch.rand(3, 224, 224) # Creates a tensor with 3 channels (e.g., RGB) and 224x224 pixels
print(f"Image-like Tensor: {image_like_tensor}\nShape: {image_like_tensor.shape}\nDimensions: {image_like_tensor.ndim}")

Image-like Tensor: tensor([[[0.6620, 0.1195, 0.7329,  ..., 0.4704, 0.6515, 0.3677],
         [0.5278, 0.4480, 0.7061,  ..., 0.9775, 0.9815, 0.4826],
         [0.8433, 0.0016, 0.0720,  ..., 0.1522, 0.5299, 0.7268],
         ...,
         [0.2445, 0.6812, 0.7293,  ..., 0.6921, 0.0436, 0.1419],
         [0.6292, 0.6360, 0.5034,  ..., 0.6232, 0.7414, 0.1346],
         [0.9181, 0.8385, 0.9106,  ..., 0.9503, 0.7196, 0.2045]],

        [[0.3174, 0.4825, 0.8144,  ..., 0.6287, 0.3615, 0.2366],
         [0.6328, 0.1155, 0.6025,  ..., 0.0085, 0.3103, 0.8732],
         [0.2819, 0.1138, 0.0036,  ..., 0.1571, 0.6205, 0.9297],
         ...,
         [0.7418, 0.4566, 0.2561,  ..., 0.5039, 0.8638, 0.9321],
         [0.7995, 0.9105, 0.0510,  ..., 0.8870, 0.6978, 0.5538],
         [0.6602, 0.7355, 0.8784,  ..., 0.1924, 0.5276, 0.5442]],

        [[0.3086, 0.5080, 0.9285,  ..., 0.9481, 0.3947, 0.3558],
         [0.0822, 0.4055, 0.4060,  ..., 0.9168, 0.7202, 0.1651],
         [0.2085, 0.8941, 0.1261,  ...,

### Zeros and Ones

`torch.zeros(shape)` and `torch.ones(shape)` create tensors filled entirely with `0`s or `1`s, useful for initializing placeholders, masks, or bias terms.


In [75]:
zero_tensor = torch.zeros(3, 4) # Creates a 3x4 tensor filled with zeros
print(f"Zero Tensor: {zero_tensor}\nShape: {zero_tensor.shape}\nDimensions: {zero_tensor.ndim}")

Zero Tensor: tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
Shape: torch.Size([3, 4])
Dimensions: 2


In [76]:
one_tensor = torch.ones(3, 4) # Creates a 3x4 tensor filled with ones
print(f"One Tensor: {one_tensor}\nShape: {one_tensor.shape}\nDimensions: {one_tensor.ndim}")

One Tensor: tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
Shape: torch.Size([3, 4])
Dimensions: 2


### Creating a Range of Tensors and Tensors-Like

- `torch.arange(start, end, step)` creates a 1D tensor of evenly spaced values.
- `torch.zeros_like(input)` / `torch.ones_like(input)` create a tensor of zeros/ones matching the shape of another tensor.


In [77]:
# Use torch.arange to create a tensor with a range of values
range_tensor = torch.arange(0, 10, 2) # Creates a tensor with values from 0 to 10 (exclusive) with a step of 2
print(f"Range Tensor: {range_tensor}\nShape: {range_tensor.shape}\nDimensions: {range_tensor.ndim}")

Range Tensor: tensor([0, 2, 4, 6, 8])
Shape: torch.Size([5])
Dimensions: 1


In [78]:
tensors_like = torch.ones_like(range_tensor) # Creates a tensor of ones with the same shape as range_tensor
print(f"Tensors Like: {tensors_like}\nShape: {tensors_like.shape}\nDimensions: {tensors_like.ndim}")

Tensors Like: tensor([1, 1, 1, 1, 1])
Shape: torch.Size([5])
Dimensions: 1


### Tensor Datatypes

**Note:** Tensor datatype issues are one of the 3 most common errors in PyTorch & deep learning:
1. Tensor is the wrong **datatype**
2. Tensor is the wrong **shape**
3. Tensor is on the wrong **device**


In [79]:
# Float32 is the default data type for PyTorch tensors. You can explicitly specify the data type when creating a tensor.
float_tensor = torch.tensor([1.0, 2.0, 3.0], 
                            dtype=None, # Specifies the data type (None --> float32, lower float means lower precision and less memory usage, int is also possible)
                            device="cuda", # Specifies the device (None --> CPU)
                            requires_grad=False)

print(f"Float32 Tensor: {float_tensor}\nShape: {float_tensor.shape}\nDimensions: {float_tensor.ndim}\nData Type: {float_tensor.dtype}")

Float32 Tensor: tensor([1., 2., 3.], device='cuda:0')
Shape: torch.Size([3])
Dimensions: 1
Data Type: torch.float32


In [80]:
# clean cache on device
torch.cuda.empty_cache()
print("CUDA cache cleared.")

CUDA cache cleared.


### Getting Information from Tensors

Every tensor carries metadata that's worth checking when debugging, most importantly its `dtype` (datatype), `shape`, `device`, and whether it `requires_grad` (needed for autograd/backpropagation).


In [81]:
# Getting information about the tensor
print(f"Tensor Device: {float_tensor.device}")
print(f"Tensor Requires Grad: {float_tensor.requires_grad}")
print(f"Tensor Data Type: {float_tensor.dtype}")
print(f"Tensor Shape: {float_tensor.shape}")

Tensor Device: cuda:0
Tensor Requires Grad: False
Tensor Data Type: torch.float32
Tensor Shape: torch.Size([3])


### Tensor Manipulations (Operations)

- Addition
- Subtraction
- Multiplication (element-wise)
- Division

In [82]:
# Tensor operations can be performed on tensors (using methods or +-*/).
# These operations can be performed element-wise or as matrix operations, depending on the operation and the shapes of the tensors involved.
init_tensor = torch.tensor([1, 2, 3])
print(f"Init Tensor: {init_tensor}")

# Addition
print(f"Addition: {init_tensor.add(10)}")

# Subtraction
print(f"Subtraction: {init_tensor.subtract(10)}")

# Division
print(f"Division: {init_tensor.divide(10)}")

# Multiplication Element-wise 
print(f"Multiplication: {init_tensor.mul(10)}") # same as init_tensor * 10

Init Tensor: tensor([1, 2, 3])
Addition: tensor([11, 12, 13])
Subtraction: tensor([-9, -8, -7])
Division: tensor([0.1000, 0.2000, 0.3000])
Multiplication: tensor([10, 20, 30])


### Matrix Multiplication (Dot Product)

**Rules**

1. **Inner dimensions** must match
   - `(3,2) @ (2,3)` → works
   - `(3,2) @ (3,2)` → won't work

2. The resulting matrix has the shape of the **outer dimensions**
   - `(3,2) @ (2,3)` → `(3,3)`
   - `(2,3) @ (3,2)` → `(2,2)`

**Fix: Using transpose**

- Use `transpose()` or `.t` to change the shape of a tensor when inner dimensions don't match. This is commonly used in linear algebra operations like matrix multiplication.
- `transpose()` returns a new tensor with the original tensor's dimensions reversed. For a 2D tensor (matrix), this swaps rows and columns.

In [83]:
print(f"Matrix Multiplication: {torch.matmul(init_tensor, init_tensor)}") # same as init_tensor.matmul(init_tensor) and torch.mm(init_tensor, init_tensor)

Matrix Multiplication: 14


In [84]:
tensor_A = torch.tensor([[1, 2],
                        [3, 4],
                        [5, 6]])

tensor_B = torch.tensor([[7, 8],
                        [9, 10],    
                        [11, 12]])

print(f"Shape of tensor_A: {tensor_A.shape}")
print(f"Shape of tensor_B: {tensor_B.shape}")

print(f"Shape of transposed tensor_B: {tensor_B.T.shape}\n")
print(f"Matrix Multiplication of tensor_A and transposed tensor_B:\n {torch.matmul(tensor_A, tensor_B.T)}")

Shape of tensor_A: torch.Size([3, 2])
Shape of tensor_B: torch.Size([3, 2])
Shape of transposed tensor_B: torch.Size([2, 3])

Matrix Multiplication of tensor_A and transposed tensor_B:
 tensor([[ 23,  29,  35],
        [ 53,  67,  81],
        [ 83, 105, 127]])


### Tensor Aggregation (min, max, mean, sum, ...)

Aggregation functions reduce a tensor down to a single summary value, such as its minimum, maximum, mean, or sum.


In [85]:
x = torch.arange(0, 100, 10) # Creates a tensor with values from 0 to 100 (exclusive) with a step of 10
print(f"Tensor x: {x}")
print(f"x shape: {x.shape}")
print(f"x dtype: {x.dtype}")
print(f"Min x: {torch.min(x)}") # Alternative: x.min()
print(f"Max x: {torch.max(x)}") # Alternative: x.max()
print(f"Mean x: {torch.mean(x.float())}") # Convert to float for mean calculation: fix dtype error
print(f"Mean x (alternative): {torch.mean(x.type(torch.float32))}")


Tensor x: tensor([ 0, 10, 20, 30, 40, 50, 60, 70, 80, 90])
x shape: torch.Size([10])
x dtype: torch.int64
Min x: 0
Max x: 90
Mean x: 45.0
Mean x (alternative): 45.0


In [86]:
# Positional min, max, and mean
print(f"Position of min x: {x.argmin()}") # Returns the index of the minimum value in the tensor
print(f"Position of max x: {x.argmax()}") # Returns the index of the maximum value in the tensor

Position of min x: 0
Position of max x: 9


### Reshaping, Stacking, Squeezing, Unsqueezing, and Permuting Tensors

- **Reshaping** — `torch.reshape(input, shape)` or `tensor.reshape(shape)`. Reshapes a tensor to a defined shape; the total number of elements must stay the same (e.g. `(2,6)` → `(3,4)` or `(12,)`, but not `(2,5)`).

- **View** — `tensor.view(shape)`. Returns a view of the original tensor in a different shape but shares the same underlying memory. Changing the view's data also changes the original tensor's data, since no new memory is allocated. Note: `view()` requires the tensor to be contiguous in memory; `reshape()` handles this automatically and is often safer.

- **Stacking** — `torch.stack(tensors, dim=0)` adds a new dimension when combining tensors; `torch.cat(tensors, dim=0)` joins tensors along an existing dimension. All tensors must have compatible shapes along the dimensions not being combined.

- **Squeezing** — `tensor.squeeze()`. Removes all dimensions of size `1` from a tensor (e.g. `(1,3,1,5)` → `(3,5)`). You can also target a specific dimension with `squeeze(dim=n)`.

- **Unsqueezing** — `tensor.unsqueeze(dim=n)`. Adds a dimension of size `1` at position `n` (e.g. `(3,5)` with `unsqueeze(dim=0)` → `(1,3,5)`).

- **Permuting** — `tensor.permute(*dims)`. Returns a *view* of the original tensor with its dimensions reordered according to the given order (e.g. a `(224,224,3)` image tensor with `permute(2,0,1)` → `(3,224,224)`, useful for switching between HWC and CHW image formats). Like `view()`, it shares the same underlying memory as the original tensor, so changes to one affect the other.

In [87]:
y = torch.arange(1., 10.) # Creates a tensor with values from 1 to 9 (exclusive) with a step of 1
print(f"Tensor y: {y}\nShape: {y.shape}\nDimensions: {y.ndim}\n")

# Add extra dimension to y
y_reshaped = y.reshape(1, 9) # The product of reshaping is equal to the original tensor size (1*9 = 9)
print(f"Reshaped Tensor y: {y_reshaped}\nShape: {y_reshaped.shape}\nDimensions: {y_reshaped.ndim}")

Tensor y: tensor([1., 2., 3., 4., 5., 6., 7., 8., 9.])
Shape: torch.Size([9])
Dimensions: 1

Reshaped Tensor y: tensor([[1., 2., 3., 4., 5., 6., 7., 8., 9.]])
Shape: torch.Size([1, 9])
Dimensions: 2


In [88]:
# Change the view of the tensor without changing its data (memory layout)

v = y.view(3, 3) # The product of reshaping is equal to the original tensor size (3*3 = 9)
print(f"View Tensor v: {v}\nShape: {v.shape}\nDimensions: {v.ndim}")

View Tensor v: tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]])
Shape: torch.Size([3, 3])
Dimensions: 2


In [89]:
# Stack tensors along an existing dimension
s = torch.stack([y, y, y, y], dim=0) # Stacks the tensors along the first dimension (dim=0)
print(f"Stacked Tensor s: {s}\nShape: {s.shape}\nDimensions: {s.ndim}")

Stacked Tensor s: tensor([[1., 2., 3., 4., 5., 6., 7., 8., 9.],
        [1., 2., 3., 4., 5., 6., 7., 8., 9.],
        [1., 2., 3., 4., 5., 6., 7., 8., 9.],
        [1., 2., 3., 4., 5., 6., 7., 8., 9.]])
Shape: torch.Size([4, 9])
Dimensions: 2


In [90]:
# Squeezing a tensor removes dimensions of size 1 from the tensor's specific dimension. For example, if a tensor has a shape of (1, 3, 1, 5), squeezing it will result in a shape of (3, 5).
print(f"Squeezed Tensor: {s.squeeze()}\nShape: {s.squeeze().shape}\nDimensions: {s.squeeze().ndim}")
# Unsqueezing a tensor adds a dimension of size 1 to the tensor's specific dimension. For example, if a tensor has a shape of (3, 5), unsqueezing it will result in a shape of (1, 3, 5).
print(f"Unsqueezed Tensor: {s.unsqueeze(0)}\nShape: {s.unsqueeze(0).shape}\nDimensions: {s.unsqueeze(0).ndim}")

Squeezed Tensor: tensor([[1., 2., 3., 4., 5., 6., 7., 8., 9.],
        [1., 2., 3., 4., 5., 6., 7., 8., 9.],
        [1., 2., 3., 4., 5., 6., 7., 8., 9.],
        [1., 2., 3., 4., 5., 6., 7., 8., 9.]])
Shape: torch.Size([4, 9])
Dimensions: 2
Unsqueezed Tensor: tensor([[[1., 2., 3., 4., 5., 6., 7., 8., 9.],
         [1., 2., 3., 4., 5., 6., 7., 8., 9.],
         [1., 2., 3., 4., 5., 6., 7., 8., 9.],
         [1., 2., 3., 4., 5., 6., 7., 8., 9.]]])
Shape: torch.Size([1, 4, 9])
Dimensions: 3


In [91]:
# Permute the dimensions of a tensor. For example, if a tensor has a shape of (2, 3, 4), permuting it with (1, 0, 2) will result in a shape of (3, 2, 4).
x_original = torch.randn(224,224,3)
print(f"Shape: {x_original.shape}\nDimensions: {x_original.ndim}")

x_permuted = x_original.permute(2, 0, 1) # Permutes the dimensions of the tensor to (3, 224, 224)
print(f"\nShape: {x_permuted.shape}\nDimensions: {x_permuted.ndim}")

Shape: torch.Size([224, 224, 3])
Dimensions: 3

Shape: torch.Size([3, 224, 224])
Dimensions: 3


### Indexing (select data from tensors)
* Indexing with PyTorch is similar to indexing with NumPy.

In [92]:
index_tensor = torch.arange(1, 10).reshape(1, 3, 3) # Creates a tensor with values from 1 to 9 (exclusive) and reshapes it to a 3x3 tensor
index_tensor, index_tensor.shape

(tensor([[[1, 2, 3],
          [4, 5, 6],
          [7, 8, 9]]]),
 torch.Size([1, 3, 3]))

In [93]:
index_tensor[0]

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])

In [94]:
index_tensor[0][0]

tensor([1, 2, 3])

In [96]:
index_tensor[0][0][0]
# use : to access all elements in a dimension

tensor(1)

In [97]:
index_tensor[:,1,1] # Accesses all elements in the first dimension, the second row, and the second column of the tensor

tensor([5])

### PyTorch Tensors & NumPy

NumPy is a widely used scientific computing library, and PyTorch tensors can interoperate with it directly.

- **NumPy array → tensor** — `torch.from_numpy(ndarray)`. Converts a NumPy array to a PyTorch tensor. By default, this shares memory with the original array, so changes to one affect the other. Note: NumPy's default dtype is `float64`, while PyTorch's default is `float32` — the converted tensor keeps `float64` unless you explicitly cast it (e.g. with `.type(torch.float32)`).

- **Tensor → NumPy array** — `tensor.numpy()`. Converts a PyTorch tensor to a NumPy array, also sharing the same underlying memory. This only works for tensors on the CPU — a GPU tensor must be moved with `.cpu()` first.

**Note:** Because the conversions share memory rather than copying it, in-place operations on either object will silently change the other. If you need an independent copy, use `.clone()` (tensor) or `.copy()` (NumPy array) before converting.

In [ ]:
# Numpy to tensor conversion
# Numpy default data type is float64, while PyTorch default data type is float32. When converting from NumPy to PyTorch, the data type will be preserved unless specified otherwise.

array = np.arange(1.0, 10.0)# Creates a NumPy array with values from 0 to 10 (exclusive)
tensor = torch.from_numpy(array)

array, tensor

(array([1., 2., 3., 4., 5., 6., 7., 8., 9.]),
 tensor([1., 2., 3., 4., 5., 6., 7., 8., 9.], dtype=torch.float64))

In [ ]:
# Tensor to Numpy conversion
tensor = torch.ones(7)
numpy_tensor = tensor.numpy() # Converts the PyTorch tensor to a NumPy array

tensor, numpy_tensor

(tensor([1., 1., 1., 1., 1., 1., 1.]),
 array([1., 1., 1., 1., 1., 1., 1.], dtype=float32))

### Reproducibility (Trying to Take the Random Out of Random)

Many operations in PyTorch use randomness (e.g. `torch.rand()`), which means results differ every time code runs. To make experiments reproducible, PyTorch uses **pseudorandom number generators** — random-looking values generated from a fixed starting point (a "seed"), so the same seed always produces the same sequence of "random" numbers.

- **Setting a seed** — `torch.manual_seed(seed)`. Sets the seed for PyTorch's random number generator, ensuring reproducible results across runs.

- **Seed scope** — `manual_seed()` only affects the *next* random operation. If you need multiple reproducible random tensors, call `torch.manual_seed(seed)` again before each one.

- **GPU randomness** — If running on a GPU, also set `torch.cuda.manual_seed(seed)` (or `torch.cuda.manual_seed_all(seed)` for multi-GPU setups), since CPU and GPU random number generators are seeded separately.

**Note:** Reproducibility also depends on the hardware and library versions used — the same seed can still produce different results across different devices (e.g. CPU vs GPU) or PyTorch versions.

### Accessing a GPU (CUDA)

PyTorch can run tensor operations on a GPU, which is significantly faster than the CPU for large-scale computations like training neural networks. CUDA is NVIDIA's platform for GPU acceleration (AMD GPUs use ROCm instead, but PyTorch exposes both through the same `cuda` API).

- **Checking availability** — `torch.cuda.is_available()`. Returns `True` if a compatible GPU is detected, `False` otherwise. Always check this before assuming a GPU is present, since code should ideally run on both CPU-only and GPU-equipped machines.

- **Device-agnostic code** — A common pattern is to set a `device` variable once:
```python
  device = "cuda" if torch.cuda.is_available() else "cpu"
```
  and then use it throughout the notebook/script, so the same code works regardless of hardware.

- **Moving tensors to a device** — `tensor.to(device)`. Moves a tensor to the specified device (CPU or GPU). Operations between tensors require them to be on the *same* device — mixing a CPU tensor with a GPU tensor will raise an error.

- **Checking device count** — `torch.cuda.device_count()`. Returns the number of GPUs available, useful when working with multiple GPUs.

**Note:** Tensors on a GPU can't be directly converted to NumPy (`.numpy()` requires the CPU) — use `.cpu()` first to move the tensor back before converting.

#### Progress Note

Stopped video at 04:00:00.
